# Module 11: Text Analytics in PostgreSQL

**ALY 6420 | Pattern matching, full-text search, string parsing, and technique selection**

*Course Lecture Notes*

## Module 11
### Moving from semi-structured data to free text
Module 10 explored JSON and arrays. A JSON key gives us a known path; a free-text email subject does not. Nevertheless, a subject may encode a campaign, discount, tracking identifier or customer concern.

The analytical workflow is: **business question → text representation → technique → validation → interpretation**.

Consider `Summer Sale: 20% Off All Scooters`. We can filter for sale messages, search for scooter-related documents, or extract the numeric discount. These are different tasks.

## Learning objectives
By the end of this lecture, you should be able to:
- use `LIKE`, `ILIKE`, `SIMILAR TO` and regular expressions;
- explain wildcard placement and index eligibility;
- construct `tsvector` and `tsquery` values and rank matches;
- create an indexed generated search vector;
- extract and cast values using `SPLIT_PART`, `POSITION`, `SUBSTRING` and `REGEXP_MATCH`;
- distinguish literal, linguistic and fuzzy search;
- validate row grain, missing values and analytical denominators.

## Principal source and database
### Principal source
Shan et al. (2025), *SQL for Data Analytics*, 4th ed., Chapter 12, **Advanced Data Types: Date, Text, and Geospatial**, pp. 240–244 (text processing and Exercise 12.2). The textbook introduces string transformations and wildcard matching through product branding.

### Working database
Use PostgreSQL 16 through DBeaver and the ZoomZoom / `sqlda` database. A temporary teaching dataset below provides reproducible examples. Run its setup in one DBeaver session and retain that session.

## First step: inspect the schema
Inspect names, types and nullability before adapting examples.

```sql
SELECT table_schema, table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_name IN ('emails', 'products')
ORDER BY table_schema, table_name, ordinal_position;
```

Reference dataset: The following eight records are adapted from the course note on Canvas.

```sql
CREATE TEMP TABLE m11_emails (
    email_id integer PRIMARY KEY, customer_id integer,
    subject text, opened boolean, sent_date date
);
INSERT INTO m11_emails VALUES
(1,101,'Summer Sale: 20% Off All Scooters This Weekend',true,'2024-06-15'),
(2,102,'Your ZoomZoom X1 Pro Order Has Shipped -- TRK-2024-8821',true,'2024-06-16'),
(3,103,'Reminder: Your Test Ride Appointment Is Tomorrow at 2PM',false,'2024-06-17'),
(4,104,'ZoomZoom Rewards: You Have Earned 500 Points',true,'2024-06-18'),
(5,105,'Important: Battery Recall Notice for X2 Cargo Models',false,'2024-06-19'),
(6,106,'Exclusive: Free Delivery on Orders Over $500 This Month',true,'2024-06-20'),
(7,107,'Summer Sale: Last Chance -- Ends This Sunday',true,'2024-06-21'),
(8,108,'Your ZoomZoom X3 Pro Has Been Delivered',true,'2024-06-22');
SELECT COUNT(*) AS rows, COUNT(DISTINCT email_id) AS unique_emails
FROM m11_emails;
```

# Part 1: Pattern Matching
## Equality versus patterns
Equality requires a complete value match. Pattern predicates test character structure. In PostgreSQL 16, `LIKE` normally distinguishes case; `ILIKE` ignores case according to locale. `%` permits any number of characters and `_` permits exactly one. Both operators match the entire input, so substring search needs surrounding `%` characters.

```sql
SELECT 'ScooterX' LIKE 'Scooter_' AS one_character,
       'Scooter12' LIKE 'Scooter_' AS two_characters,
       'Scooter' LIKE 'Scooter%' AS zero_characters,
       'Summer Sale' LIKE '%sale%' AS case_sensitive,
       'Summer Sale' ILIKE '%sale%' AS case_insensitive;
```

Expected: true, false, true, false, true.

| Pattern | Meaning |
|---|---|
| `Summer%` | prefix |
| `%Scooters` | suffix |
| `%Sale%` | substring |
| `X_ Pro` | exactly one character after X |

```sql
SELECT email_id, subject FROM m11_emails WHERE subject LIKE 'Summer%' ORDER BY email_id;
```

```sql
SELECT email_id, subject FROM m11_emails WHERE subject NOT ILIKE '%sale%' ORDER BY email_id;
```

The first query finds IDs 1 and 7; the second finds 2, 3, 4, 5, 6, 8. A null subject is excluded by both a positive and negative predicate unless handled explicitly.

### Example 11.1: Support ticket routing

**Goal and context.** ILIKE ignores case; the trailing wildcard permits any suffix.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT id FROM (VALUES (1,'Battery warning'),(2,'battery low'),(3,'Brake noise')) t(id,msg) WHERE msg ILIKE 'battery%' ORDER BY id;
```

**Expected result:** 1, 2.

**Derived practice:** Change to LIKE with lowercase battery. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Only 2.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Literal wildcards and word boundaries
A percentage sign in an email is data; in a LIKE pattern it is syntax. Choose an explicit escape character. Substrings are not necessarily whole words (for example, `wholesale` contains `sale`).

```sql
SELECT 'Save 20%' LIKE '%20!%%' ESCAPE '!' AS literal_percent,
       'wholesale' ILIKE '%sale%' AS substring_match,
       'wholesale' ~* '\msale\M' AS whole_word_match;
```

## SIMILAR TO
This operator combines SQL wildcards with alternation and repetition. It also checks the entire input. Unlike POSIX regex, a dot is not its general single-character wildcard.

```sql
SELECT email_id, subject FROM m11_emails WHERE subject SIMILAR TO '%(Sale|Recall)%' ORDER BY email_id;
```

Expected IDs: 1, 5, 7. Prefer the simplest readable predicate that expresses the requirement.

### Example 11.2: Warehouse bin format

**Goal and context.** SIMILAR TO matches the whole string.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT code FROM (VALUES ('A-12'),('B-99'),('C-10'),('A-123')) t(code) WHERE code SIMILAR TO '(A|B)-[0-9]{2}' ORDER BY code;
```

**Expected result:** A-12, B-99.

**Derived practice:** Allow C as well. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

A-12, B-99, C-10.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## POSIX regular expressions
| Syntax | Meaning |
|---|---|
| `~`, `~*` | case-sensitive / insensitive match |
| `!~`, `!~*` | negated match |
| `^`, `$` | start / end anchor |
| `[0-9]`, `\d` | digit patterns |
| `+`, `*`, `?` | one or more, zero or more, optional |
| `{4}` | exactly four repetitions |
| `(a|b)` | alternatives |

Regex searches can match part of a string without surrounding wildcards. Parentheses also capture substrings for extraction.

```sql
SELECT email_id, subject
FROM m11_emails
WHERE subject ~ '[0-9]+%';
-- Expected: ID 1
SELECT email_id, subject
FROM m11_emails
WHERE subject ~ '\$[0-9]+';
-- Expected: ID 6
```

Dollar signs must be escaped when used literally in a regex. The unescaped `$` asserts an end position, so `$[0-9]+` cannot match. These examples assume PostgreSQL’s default standard-conforming strings setting.

```sql
SELECT regexp_match('Code TRK-2024-8821', 'TRK-[0-9]{4}-[0-9]{4}') AS full_match,
       regexp_match('Save 20%', '([0-9]+)%') AS capture,
       regexp_replace('Save   today', '[[:space:]]+', ' ', 'g') AS cleaned;
```

Expected arrays contain `TRK-2024-8821` and `20`, respectively. The replacement produces `Save today`. The `g` flag replaces every match. `regexp_matches` (plural) returns a set; with `g` it may produce multiple rows per source record.

### Example 11.3: Mask identifiers in telemetry

**Goal and context.** The global flag replaces both tokens; this narrow regex is not comprehensive de-identification.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT regexp_replace('unit=AB12 unit=CD34','unit=[A-Z]{2}[0-9]{2}','unit=REDACTED','g');
```

**Expected result:** unit=REDACTED unit=REDACTED.

**Derived practice:** Remove g. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Only the first token is replaced.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Wildcards and performance
A sorted index can narrow a constant prefix to a range of keys. A leading wildcard does not supply that starting range.

| Predicate | Ordinary prefix-index opportunity |
|---|---|
| `subject LIKE 'Summer%'` | possible |
| `subject LIKE '%Summer%'` | not a direct B-tree prefix lookup |
| `subject LIKE '%Scooters'` | not a direct B-tree prefix lookup |
| `subject ~ '^Summer'` | possible |

Collation and operator class matter; `text_pattern_ops` supports prefix matching for text under many non-C locales. A planner may still choose a sequential scan for small tables. Trigram indexes can help substring and many regex searches.

```sql
CREATE INDEX m11_subject_prefix_idx ON m11_emails (subject text_pattern_ops);
EXPLAIN SELECT * FROM m11_emails WHERE subject LIKE 'Summer%';
EXPLAIN SELECT * FROM m11_emails WHERE subject LIKE '%Sale%';
```

Do not infer performance from eight rows. Inspect the access path on representative data in Module 12. 

### Example 11.4: Prefix versus substring

**Goal and context.** A prefix has a bounded beginning; a substring does not.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT label, label LIKE 'hub%' AS prefix, label LIKE '%hub%' AS anywhere FROM (VALUES ('hub-east'),('microhub'),('depot')) t(label);
```

**Expected result:** true/true; false/true; false/false.

**Derived practice:** Try %east. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Only hub-east matches; ordinary B-tree prefix bounds do not apply.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 2: Full-Text Search
## From characters to lexemes
Full-text search works with normalized tokens rather than arbitrary substrings. The English configuration removes common stop words and reduces many inflected forms to a common representation. Positions are retained for phrase matching. It is not semantic or typo-tolerant search.

```sql
SELECT to_tsvector('english', 'The scooters are on sale') AS document;
```

Expected: `'sale':5 'scooter':2` — words at positions 1, 3 and 4 are removed.

```text
Raw text → tokenize → dictionary/normalization → tsvector
Search input → normalize → tsquery
                  tsvector @@ tsquery → match
```
Use the same configuration for document and query. The `simple` configuration lowercases tokens without English stemming.

```sql
SELECT to_tsvector('english', 'Scooters on sale') @@
       plainto_tsquery('english', 'scooter sale') AS matched;
```

## Constructing search queries
| Function | Input intention |
|---|---|
| `to_tsquery` | Explicit operators `&`, `|`, `!`, `<->` |
| `plainto_tsquery` | All surviving keywords required |
| `phraseto_tsquery` | Ordered lexemes with phrase distances |
| `websearch_to_tsquery` | Search-box syntax, quoted phrases, OR, minus |

Phrase search uses normalized lexemes and is not necessarily a verbatim character match.

```sql
SELECT to_tsquery('english', 'scooter | delivery') AS explicit_or,
       plainto_tsquery('english', 'summer scooter') AS all_terms,
       phraseto_tsquery('english', 'summer sale') AS phrase,
       websearch_to_tsquery('english', '"summer sale" -recall') AS web_query;
```

```sql
SELECT email_id, subject FROM m11_emails
WHERE to_tsvector('english', subject) @@ to_tsquery('english', 'scooter | delivery')
ORDER BY email_id;
```

Expected IDs: 1 and 6. `Delivered` and `delivery` need not produce the same lexeme. Inspect `to_tsvector` rather than assuming every related word matches.

### Example 11.5: Search maintenance manuals

**Goal and context.** The same text configuration normalizes documents and queries.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT id FROM (VALUES (1,'Replacing filters'),(2,'Filter replacement'),(3,'Brake inspection')) t(id,doc) WHERE to_tsvector('english',doc) @@ plainto_tsquery('english','filter') ORDER BY id;
```

**Expected result:** 1, 2.

**Derived practice:** Search for brake. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

3.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Ranking relevant documents
Matching decides eligibility; ranking orders the eligible results. Rank values are query-dependent scores, not probabilities or open rates. Use a stable tie-breaker.

```sql
WITH docs AS (
    SELECT email_id, subject, to_tsvector('english', COALESCE(subject,'')) AS doc
    FROM m11_emails
), search AS (
    SELECT to_tsquery('english', 'summer | scooter | sale') AS query
)
SELECT email_id, subject, ts_rank(doc, query) AS relevance
FROM docs CROSS JOIN search
WHERE doc @@ query
ORDER BY relevance DESC, email_id;
```

Expected matched IDs: 1 and 7. Inspect scores rather than assuming precise values. Normalization option 32 rescales rank using rank/(rank+1), but does not make it a probability.

## Weighting sections and highlighting
A subject can carry greater weight than a body. This example uses literal text so that no unverified schema column is required.

```sql
WITH doc AS (
 SELECT setweight(to_tsvector('english','Scooter sale'),'A') ||
        setweight(to_tsvector('english','Free delivery this weekend'),'D') AS v
)
SELECT ts_rank(v, plainto_tsquery('english','scooter')) AS subject_rank,
       ts_rank(v, plainto_tsquery('english','delivery')) AS body_rank
FROM doc;
SELECT ts_headline('english','Scooters are available this weekend',
                   plainto_tsquery('english','scooter'));
```

Use `COALESCE` for nullable source columns when adapting the weighted expression. Highlighted text is for display, not a safe HTML sanitizer.

### 11.6: Lexical retrieval for an AI assistant

**Goal and context.** Retrieve candidates before generation; lexical relevance is not factual accuracy or semantic embedding similarity.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH docs(id,body) AS (VALUES (1,'reset password'),(2,'reset password reset password'),(3,'change address')), q AS (SELECT plainto_tsquery('english','reset password') AS query) SELECT id,ts_rank(to_tsvector('english',body),query) AS score FROM docs CROSS JOIN q WHERE to_tsvector('english',body) @@ query ORDER BY score DESC,id;
```

**Expected result:** 2 matching rows, IDs 1 and 2; inspect scores.

**Derived practice:** Search for address instead. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Only ID 3. In a real RAG system enforce authorization before retrieval.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## GIN indexing
A GIN index associates lexemes with matching row references. Store a generated vector to avoid recomputing it for each search. A matching expression index is another option. The following DDL affects only the temporary teaching table.

```sql
ALTER TABLE m11_emails ADD COLUMN subject_tsv tsvector
GENERATED ALWAYS AS (to_tsvector('english', COALESCE(subject,''))) STORED;
CREATE INDEX m11_subject_tsv_idx ON m11_emails USING GIN (subject_tsv);
SELECT email_id, subject FROM m11_emails
WHERE subject_tsv @@ plainto_tsquery('english','summer sale')
ORDER BY email_id;
EXPLAIN SELECT * FROM m11_emails
WHERE subject_tsv @@ plainto_tsquery('english','summer sale');
```

Expected IDs: 1 and 7. The vector refreshes when its source changes. The index adds storage and write overhead; ranking still requires work on the matching rows. Do not alter shared `emails` tables without authorization.

### Example 11.7: Index a local document collection

**Goal and context.** The indexed expression matches the predicate.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
CREATE TEMP TABLE m11_manual AS SELECT i AS id, CASE WHEN i%100=0 THEN 'hydraulic leak' ELSE 'routine inspection' END AS body FROM generate_series(1,10000) i;
CREATE INDEX ON m11_manual USING gin(to_tsvector('english',body));
ANALYZE m11_manual;
EXPLAIN SELECT id FROM m11_manual WHERE to_tsvector('english',body) @@ plainto_tsquery('english','hydraulic');
SELECT count(*) FROM m11_manual WHERE to_tsvector('english',body) @@ plainto_tsquery('english','hydraulic');
DROP TABLE m11_manual;
```

**Expected result:** 100 matches; plan varies.

**Derived practice:** Search routine instead. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

9900 matches; a sequential scan may be cheaper.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 3: String Parsing and Transformation
## Basic string functions 
| Function | Purpose |
|---|---|
| `length`, `lower`, `upper` | measure or standardize |
| `replace`, `reverse`, `repeat` | transform |
| `trim`, `ltrim`, `rtrim` | remove boundary spaces by default |
| `left`, `right`, `substring`, `substr` | extract characters |
| `position`, `strpos` | locate a substring |
| `split_part` | extract a delimited field |



```sql
WITH candidates(model) AS (VALUES ('Model Chi'),('Model Epsilon'),('Model Extra Long Name'))
SELECT model, replace(model,'Model','Super Model') AS new_model,
       length(replace(model,'Model','Super Model')) <= 20 AS fits_poster
FROM candidates;
```

Expected: first two fit; the third does not. In `sqlda`, substitute the verified `products` table. Compare `model LIKE 'Model%'` with `left(model,5) = 'Model'`: the logical output can agree while index eligibility differs.

## Delimiter and position-based extraction

```sql
SELECT split_part('Order: TRK-2024-8821: Shipped', ':', 2) AS raw_part,
       trim(split_part('Order: TRK-2024-8821: Shipped', ':', 2)) AS clean_part,
       position('TRK' IN 'Your Order TRK-2024-8821 Has Shipped') AS start_pos,
       substring('Your Order TRK-2024-8821 Has Shipped' FROM 12 FOR 13) AS tracking;
```

Expected start position: 12. Tracking identifier length: 13. Positions count characters starting at one. Prefer dynamic positions to hard-coded offsets.

```sql
SELECT email_id, trim(split_part(subject, ':', 1)) AS prefix,
       trim(substring(subject FROM position(':' IN subject)+1)) AS description
FROM m11_emails WHERE position(':' IN subject)>0
ORDER BY email_id;
```

Expected IDs: 1, 3, 4, 5, 6, 7. The guard matters: `POSITION` returns zero if absent, and `SPLIT_PART` returns an empty string for an out-of-range field. Without the guard, a subject without a colon would be treated as a prefix.

### Example 11.8: Parse logistics codes

**Goal and context.** Delimiter extraction does not validate the format.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT split_part(code,'/',1) AS depot, split_part(code,'/',2) AS aisle FROM (VALUES ('NW/07'),('SE/12'),('BAD')) t(code);
```

**Expected result:** NW/07, SE/12, BAD/empty string.

**Derived practice:** Convert empty aisle to NULL. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Use NULLIF(split_part(code,'/',2),'').

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Regular expression extraction and casting

```sql
SELECT email_id,
       (regexp_match(subject,'TRK-[0-9]{4}-[0-9]{4}'))[1] AS tracking_id,
       (regexp_match(subject,'([0-9]+)%'))[1]::integer AS discount_pct
FROM m11_emails ORDER BY email_id;
```

ID 1 has discount 20; ID 2 has tracking ID `TRK-2024-8821`; other extracted values are null. With capture groups, array elements correspond to groups; without groups, element 1 contains the whole match. Validate numeric range and format before casting external data. The discount regex above assumes a whole-number percentage; it can misread decimal percentages.

```sql
SELECT s, (regexp_match(s, '([0-9]+(?:\.[0-9]+)?)%'))[1]::numeric AS discount
FROM (VALUES ('Save 12.5%'), ('Save 20%'), ('No discount mentioned'), (NULL)) AS t(s);
```

Expected: 12.5, 20, NULL, NULL. Missing is not the same as zero.

### Example 11.9: Safely parse sensor values

**Goal and context.** Guard the cast; a valid zero is distinct from absent data.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT raw, CASE WHEN raw ~ '^[0-9]+([.][0-9]+)?$' THEN raw::numeric END AS reading FROM (VALUES ('18.5'),('offline'),('0'),(NULL)) t(raw);
```

**Expected result:** 18.5, NULL, 0, NULL.

**Derived practice:** Allow negative readings. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Add -? immediately after ^.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 4: Technique Selection
| Analytical intent | First tool | Limitation to check |
|---|---|---|
| Exact identifier | `=` | missing or inconsistent data |
| Known prefix or substring | `LIKE` / `ILIKE` | case, word boundaries, indexes |
| Variable character structure | regex | escaping and overmatching |
| Ranked keyword retrieval | full-text search | language, stemming, missing terms |
| Extract a value | split, position, substring, regex | malformed formats and casts |
| Approximate spelling | `pg_trgm` | threshold and false matches |

**Decision rule:** If the desired result is a new attribute, parse. If it is a subset of records, decide whether character structure, linguistic relevance or approximate spelling defines membership.

## Fuzzy matching (optional extension)
Trigram similarity compares shared three-character sequences, with word-boundary padding. It can identify spelling variants, but not reliably identify synonyms or semantic equivalence. Scores range from zero to one. Extension installation requires appropriate privileges; only run in an authorized database.

```sql
-- Optional, not required for the completion check:
CREATE EXTENSION IF NOT EXISTS pg_trgm;
SELECT similarity('scooter','scootr') AS spelling_similarity;
CREATE INDEX m11_subject_trgm_idx ON m11_emails USING GIN (subject gin_trgm_ops);
```

The misspelling should yield a partial similarity, not a perfect match. Trigram indexes can also support many non-prefix LIKE and regex predicates; very short patterns may benefit little.

### Example 11.10: Choose a matching method

**Goal and context.** Fuzzy matching is unsuitable for unique equipment identifiers; exact and family search answer different questions.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT 'AX-42' = 'AX-43' AS exact_match, 'AX-42' LIKE 'AX-%' AS family_match;
```

**Expected result:** false, true.

**Derived practice:** Match only the AX-42 asset. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Use equality, not similarity.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 5: Applied Analytics
## Do sale subjects have higher open rates?
Business question: compare engagement between subjects mentioning a sale and other subjects. The unit of observation is one email. Unknown open status should not silently become a non-open. Unknown subjects receive their own category.

```sql
WITH classified AS (
 SELECT email_id, opened,
        CASE WHEN subject IS NULL THEN 'unknown subject'
             WHEN subject ILIKE '%sale%' THEN 'sale mention'
             ELSE 'other' END AS segment
 FROM m11_emails
)
SELECT segment, COUNT(*) AS emails,
       COUNT(opened) AS known_status,
       COUNT(*) FILTER (WHERE opened IS TRUE) AS opens,
       ROUND(100.0 * COUNT(*) FILTER (WHERE opened IS TRUE)
             / NULLIF(COUNT(opened),0),2) AS open_rate_pct
FROM classified GROUP BY segment ORDER BY segment;
```

Expected: sale mention 2/2 (100%); other 4/6 (66.67%). Total = 8 emails, 6 opens. This is illustrative, not evidence of a causal effect: timing, recipients and campaign selection could confound the comparison. An email-level rate is not a unique-customer rate.

## Adapt to ZoomZoom
After verifying schema, try the following read-only query on the real database. If `opened` is not Boolean, adapt the filter and denominator to its actual encoding.

```sql
SELECT email_subject, opened
FROM emails
WHERE email_subject ILIKE '%sale%'
LIMIT 20;
```

Compare literal matches with full-text search on the same subjects. Inspect disagreements. If a body column exists, repeat the search on a weighted combined document using its verified name. Do not expose private message bodies in public notebook outputs.

### Example 11.11: Support queue prioritization

**Goal and context.** Separate missing text from nonmatching text.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH tickets(id,msg,resolved) AS (VALUES (1,'urgent leak',true),(2,'urgent fault',false),(3,'routine check',true),(4,NULL,false)) SELECT count(*) FILTER (WHERE msg ILIKE '%urgent%') AS urgent, count(*) FILTER (WHERE msg IS NULL) AS unknown FROM tickets;
```

**Expected result:** 2 urgent, 1 unknown.

**Derived practice:** Count unresolved urgent tickets. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

1; add AND NOT resolved to the filter.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 6: Common Errors and AI-Generated SQL Critique
| Error | Consequence | Repair |
|---|---|---|
| `LIKE '%sale%'` for all cases | misses uppercase variants | use `ILIKE` |
| `LIKE '%20%%'` for literal percent | wildcard overmatches | explicit escape or regex |
| `position(...) IS NOT NULL` | includes absent delimiter | test `> 0` |
| treating `regexp_match` as scalar text | wrong type | select array element |
| `to_tsquery('english','sale OR scooter')` | invalid query syntax | use `|` |
| assuming `ts_rank` is a probability | misleading interpretation | treat as relative score |
| assuming GIN guarantees an index scan | incorrect performance claim | inspect EXPLAIN |

### Critique exercise
An AI assistant proposes the following query to measure the average discount across promotional emails:

```sql
-- Deliberately flawed; critique before running.
SELECT AVG(COALESCE((regexp_match(subject, '([0-9]+)%'))[1]::integer,0))
FROM m11_emails
WHERE subject LIKE '%sale%';
```

Identify the case error, the treatment of missing percentages, and the assumption that all sale messages advertise discounts. Repair the query to compute the mean of explicitly advertised whole-number discounts among sale subjects; report the eligible count. Expected reference result: mean 20, count 1. Explain how decimal percentages would change your parser.

# Part 7: Practice Set and Knowledge Checks
## Completion-check preparation
Use `m11_emails` or the Canvas `ref_emails` CTE. Write your own queries before consulting the validation targets. This is formative practice, not a graded lab.

1. Retrieve subjects beginning with Summer; then find sale in any case and its negation.
2. Detect a number followed by `%`, extract the digits, and cast them to integer.
3. Find documents matching scooter, rank the results, and then search for scooter OR delivery.
4. Extract colon-delimited prefixes and the tracking identifier; group by whether a colon is present.
5. Choose and justify a technique for subjects starting with Your ZoomZoom, ranked summer scooter promotion search, distinct prefixes, and literal dollar amounts.

### Validation targets
| Task | Expected result |
|---|---|
| Summer prefix / case-insensitive sale | IDs 1, 7 |
| Negative sale | IDs 2, 3, 4, 5, 6, 8 |
| Discount | ID 1; 20 |
| Scooter | ID 1 |
| Scooter OR delivery | IDs 1, 6 |
| Tracking | ID 2; TRK-2024-8821 |
| Colon grouping | 6 present, 2 absent |
| Distinct colon prefixes | 5 |
| Your ZoomZoom prefix | IDs 2, 8 |
| Dollar amount | ID 6 |

For summer scooter promotion, an all-term query yields no match because promotion is absent. Explain why an OR query can return candidates and how ranking changes interpretation. No exact rank values are required.

## Knowledge check
1. Why can `ILIKE '%sale%'` match wholesale?
2. What is the difference between `POSITION` returning 0 and `REGEXP_MATCH` returning NULL?
3. Why does a scooter keyword query match Scooters but not necessarily a related campaign lacking that word?
4. Why might a sequential scan be chosen despite an index?
5. How could `regexp_matches(..., 'g')` affect the result grain?

### Answer checkpoint
1. It tests a substring rather than word boundaries.
2. Zero is a valid numeric sentinel, while NULL represents an absent result.
3. Full-text search uses normalized lexemes, not inferred topic associations.
4. The estimated cost can favor a scan, especially on a small relation.
5. Multiple matches can generate multiple output rows per input row.

## Discussion 
Choose a professional context such as survey responses, CRM notes, model logs or moderation audit messages. Specify the entity, field, analytical question, chosen technique and why the alternatives are inappropriate. Discuss false positives, false negatives and validation.

# Summary
- Character patterns, lexical relevance and structured extraction are separate analytical operations.
- Choose case behavior and wildcard placement deliberately.
- Search vectors and queries must use compatible normalization; ranking orders eligible documents.
- Guard missing delimiters and failed matches before parsing and casting.
- Validate grain, denominator and expected matches before interpreting outcomes.
- Use EXPLAIN rather than assuming an index will be selected.

## References
- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.). Packt. Chapter 12, pp. 240–244.
- Northeastern University College of Professional Studies. (2026). ALY 6420: Introduction to Querying. Canvas Module 11.
- PostgreSQL Global Development Group. [9.4 String Functions](https://www.postgresql.org/docs/16/functions-string.html); [9.7 Pattern Matching](https://www.postgresql.org/docs/16/functions-matching.html); [12 Full Text Search](https://www.postgresql.org/docs/16/textsearch.html); [12.3 Controlling Text Search](https://www.postgresql.org/docs/16/textsearch-controls.html); [11.2 Index Types](https://www.postgresql.org/docs/16/indexes-types.html); [F.35 pg_trgm](https://www.postgresql.org/docs/16/pgtrgm.html).
- Additional reading assigned in Canvas: Batuigas, K. (2021). [Postgres Full-Text Search: A Search Engine in a Database](https://www.crunchydata.com/blog/postgres-full-text-search-a-search-engine-in-a-database).
